# Case Study 02 — Urban Heat Interpolation 案例二：都市氣溫熱力圖
### From scattered temperature points to a continuous surface 從離散溫度點到連續空間場

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrewwangarchnycu/gis-open-data-workshop-2026/blob/main/case-studies/02-urban-heat-interpolation/notebook.ipynb)

Extends the buffer/interpolation logic from [Lesson 04](../../lessons/04-qgis-basics/) and [Lesson 05](../../lessons/05-computational-gis/) to a new spatial operation: turning a handful of point measurements into a continuous heat surface with **IDW (Inverse Distance Weighting)**.
延伸[課程 04](../../lessons/04-qgis-basics/)與[課程 05](../../lessons/05-computational-gis/)中的緩衝區／內插邏輯，示範一種新的空間運算：以 **IDW（反距離加權法）** 將少數點狀測量值轉換為連續的溫度場。

**Difficulty 難度**: intermediate — new concept (spatial interpolation), no account/registration needed for the default data source. 中階——內插是新概念，但預設資料來源不需要註冊帳號。

## Step 0 — The research question 研究問題

**Question 問題**: *At one moment in time, how does air temperature vary across the city — and where might heat pool?* 在同一時刻，全市氣溫如何變化——哪裡可能是熱點？

**Required variables 所需變數**: temperature readings at multiple known locations, spread across the study area 研究範圍內多個已知位置的氣溫觀測值

**Data source — two options 資料來源—兩種選擇**:

| | Primary (used below) 主要（本筆記本採用） | Advanced alternative 進階替代方案 |
|---|---|---|
| Source 來源 | [Open-Meteo API](https://open-meteo.com) | [中央氣象署開放資料平台](https://opendata.cwa.gov.tw) 自動氣象站觀測資料 |
| Registration 註冊 | **None — free, no API key** 免申請、免金鑰 | Free account + personal API key (a few minutes) 免費註冊會員取得專屬 API Key |
| What you get 取得內容 | Modeled weather at *any* coordinate you choose (a "virtual sensor network") 任意座標的模擬氣象資料（可自建「虛擬感測網」） | Real physical station observations only, at fixed station locations 真實測站觀測值，僅限固定測站位置 |
| Best for 適合情境 | Quick, dense, no-friction workshops — exactly this notebook 快速、免門檻的工作坊情境——本筆記本的情況 | Research requiring authoritative, station-verified readings 需要官方認證觀測值的研究 |

> ⚠️ **Caveat 注意事項**: an earlier version of this workshop referenced 民生公共物聯網 (Civil IoT Taiwan) as a temperature-sensor source. That government program formally concluded on 2025-12-31 and its old `sta.ci.taiwan.gov.tw` API endpoint may no longer work — use the CWA source above instead if you need real physical stations.
> 本工作坊先前版本曾參考「民生公共物聯網」作為溫度感測資料來源。該計畫已於 2025 年 12 月 31 日正式結束，舊有 `sta.ci.taiwan.gov.tw` API 網址可能已失效——如需真實實體測站資料，請改用上方的中央氣象署來源。

In [ ]:
!pip install requests scipy geopandas shapely matplotlib pandas folium branca -q

In [ ]:
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
import folium
import branca.colormap as cm

print("ready")

> **Note on chart text 圖表文字說明**: plot titles/legends are kept in English because Colab's default font can't render Chinese glyphs (see [`resources/python/README.md`](../../resources/python/README.md) for a font fix). Explanations stay bilingual.
> 圖表標題／圖例保持英文，因 Colab 預設字型無法顯示中文（字型修正方式見 [`resources/python/README.md`](../../resources/python/README.md)）；說明文字維持雙語。

## Step 1 — Build a virtual sensor grid 建立虛擬感測網格

We don't need physical sensors — we choose a grid of coordinates ourselves and ask Open-Meteo for the modeled current temperature at each one. This is honest about what it is: **model output sampled at chosen points**, not physical sensor hardware. It's still real, live atmospheric data (not invented), just not from a physical sensor network.
不需要實體感測器——我我們自行選定一組座標網格，向 Open-Meteo 查詢每個座標當下的模擬氣溫。這裡要誠實說明其性質：這是**在指定點位取樣的模型輸出**，而非實體感測硬體。它仍然是真實、即時的大氣資料（非憑空捏造），只是並非來自實體感測網路。

In [ ]:
# A 6x6 grid of points over central Taipei (swap in your own city's bounding box)
# 6x6 網格點位覆蓋台北市中心（可換成你研究城市的邊界範圍）
LAT_MIN, LAT_MAX = 24.98, 25.08
LON_MIN, LON_MAX = 121.48, 121.58
N = 6

lats = np.linspace(LAT_MIN, LAT_MAX, N)
lons = np.linspace(LON_MIN, LON_MAX, N)
grid_lat, grid_lon = np.meshgrid(lats, lons)
sensor_lats = grid_lat.ravel()
sensor_lons = grid_lon.ravel()
print(f"{len(sensor_lats)} virtual sensor points")

## Step 2 — Query Open-Meteo (or fall back to offline sample) 查詢 Open-Meteo（或改用離線範例）

Open-Meteo accepts **comma-separated lists** of latitudes/longitudes in one request — up to 100 points per call, no key required. If the sandbox has no network access, this falls back to a synthetic-but-plausible temperature field so the notebook still runs end-to-end.
Open-Meteo 接受在單一請求中傳入**以逗號分隔的多組經緯度**——每次最多 100 點，且不需金鑰。若沙盒環境無網路連線，會改用一組合理但為合成的氣溫場，讓筆記本仍可完整執行。

In [ ]:
def fetch_open_meteo_temps(lats, lons):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": ",".join(f"{v:.4f}" for v in lats),
        "longitude": ",".join(f"{v:.4f}" for v in lons),
        "current": "temperature_2m",
    }
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    # Open-Meteo returns one object per point when multiple coordinates are passed
    if isinstance(data, list):
        temps = [d["current"]["temperature_2m"] for d in data]
    else:
        temps = [data["current"]["temperature_2m"]]
    return np.array(temps)

try:
    temps = fetch_open_meteo_temps(sensor_lats, sensor_lons)
    source = "Open-Meteo (live)"
except Exception as e:
    print("Live fetch failed (expected without network access) — using offline sample instead.")
    print("Error 錯誤:", e)
    # Plausible synthetic field: warmer toward the urbanized south-east corner of the grid
    # 合成範例場：越靠近網格東南（都市化較密集）方向氣溫越高，僅供離線示範
    rng = np.random.default_rng(7)
    base = 29.0
    gradient = (sensor_lats - LAT_MIN) * -6 + (sensor_lons - LON_MIN) * 4
    temps = base + gradient + rng.normal(0, 0.3, len(sensor_lats))
    source = "offline synthetic sample (illustrative only) 離線合成範例（僅供示範）"

source_label = "Open-Meteo (live)" if source.startswith("Open-Meteo") else "offline synthetic sample"
print("Source 資料來源:", source)
sensors = gpd.GeoDataFrame(
    {"temperature_c": temps},
    geometry=[Point(xy) for xy in zip(sensor_lons, sensor_lats)],
    crs="EPSG:4326",
)
sensors.head()

**Expected output 預期輸出**: a table of 36 points with a `temperature_c` column and point geometry — either live values (in Colab) or the labeled offline sample.
一張含 36 個點位的表格，含 `temperature_c` 欄位與點幾何——可能是即時數值（在 Colab 中）或標示清楚的離線範例。

## Interactive check — sensor points on a real map 互動檢查：感測點於真實地圖上

Before interpolating, look at the raw points on an actual basemap — sanity-checking data on a real map catches obvious problems (a point in the wrong city, a wildly off reading) before they propagate into the interpolated surface.
在內插之前，先在真實底圖上檢視原始點位——用真實地圖檢查資料，能在錯誤（例如點位落在錯誤城市、讀數明顯異常）擴散進內插結果之前及早發現。

In [ ]:
colormap = cm.LinearColormap(colors=["blue", "yellow", "red"],
                              vmin=float(temps.min()), vmax=float(temps.max()),
                              caption="Temperature (deg C)")

m = folium.Map(location=[sensor_lats.mean(), sensor_lons.mean()], zoom_start=11, tiles="CartoDB positron")
for lat, lon, t in zip(sensor_lats, sensor_lons, temps):
    folium.CircleMarker(
        location=[lat, lon],
        radius=8,
        color=colormap(t),
        fill=True,
        fill_color=colormap(t),
        fill_opacity=0.9,
        popup=f"{t:.1f} deg C",
    ).add_to(m)
colormap.add_to(m)
m  # displays inline in Colab 於 Colab 中直接顯示

## Step 3 — IDW interpolation: points → continuous surface 反距離加權內插：從點到連續場

**Why 為什麼**: a handful of point readings can't answer "what's the temperature *between* the points?" IDW estimates unknown locations as a distance-weighted average of nearby known points — closer points count more. This is the computational equivalent of a QGIS **Heatmap** or **IDW Interpolation** tool.
少數幾個點狀讀數無法回答「點與點之間的溫度是多少？」IDW 內插法會以已知點的距離加權平均，估算未知位置的數值——距離越近，權重越高。這是 QGIS **Heatmap** 或 **IDW 內插**工具的運算式等價版本。

In [ ]:
# Build a fine regular grid to interpolate onto 建立要內插的精細網格
grid_x, grid_y = np.mgrid[LON_MIN:LON_MAX:200j, LAT_MIN:LAT_MAX:200j]

# scipy's griddata with method="linear" approximates IDW-like smooth interpolation;
# for a textbook IDW you can swap in a manual inverse-distance-weighted average instead.
# scipy 的 griddata（method="linear"）可產生類似 IDW 的平滑內插結果；
# 若要嚴格的教科書版 IDW，可自行改寫為手動反距離加權平均。
surface = griddata(
    points=np.column_stack([sensor_lons, sensor_lats]),
    values=temps,
    xi=(grid_x, grid_y),
    method="linear",
)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(
    surface.T, extent=(LON_MIN, LON_MAX, LAT_MIN, LAT_MAX),
    origin="lower", cmap="inferno", aspect="auto",
)
ax.scatter(sensor_lons, sensor_lats, c=temps, cmap="inferno",
           edgecolor="white", linewidth=0.8, s=60)
plt.colorbar(im, ax=ax, label="Temperature (deg C)")
ax.set_title(f"Interpolated temperature surface — {source_label}")
plt.show()

## Optional — Kriging instead of IDW 選用：以 Kriging 取代 IDW

Kriging is a more statistically rigorous interpolation method that also estimates uncertainty, at the cost of more setup. Try it if you want to go deeper.
Kriging 是統計上更嚴謹的內插方法，還能估計不確定性，但設定較複雜。想深入研究可以試試看。

In [ ]:
try:
    from pykrige.ok import OrdinaryKriging
    OK = OrdinaryKriging(sensor_lons, sensor_lats, temps, variogram_model="linear", verbose=False)
    krige_surface, krige_var = OK.execute("grid", np.linspace(LON_MIN, LON_MAX, 100), np.linspace(LAT_MIN, LAT_MAX, 100))
    plt.figure(figsize=(7, 6))
    plt.imshow(krige_surface, extent=(LON_MIN, LON_MAX, LAT_MIN, LAT_MAX), origin="lower", cmap="inferno")
    plt.colorbar(label="Temperature (deg C, kriged)")
    plt.title("Kriged temperature surface")
    plt.show()
except ImportError:
    print("pykrige not installed — run `!pip install pykrige -q` first if you want to try this cell.")

## Research interpretation exercise 研究詮釋練習

- **What? 是什麼？** _______ (where is the surface hottest / coolest?)
- **Where? 在哪裡？** _______ (what's actually located there — dense built-up area? park? river?)
- **Why? 為什麼？** _______ (surface materials, building density, vegetation, wind corridors?)
- **So what? 所以呢？** _______ (what design or policy question does this raise?)

**Next steps 接下來**:
1. Change the grid resolution (`N`) or bounding box to your own study area. 修改網格解析度（`N`）或邊界範圍為你的研究區域。
2. Register for a free [CWA API key](https://opendata.cwa.gov.tw) to swap in real station observations instead of modeled values. 註冊免費的[中央氣象署 API 金鑰](https://opendata.cwa.gov.tw)，改用真實測站觀測值。
3. Take the result into [Lesson 07 — Research Map Design](../../lessons/07-research-map-design/).